# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rowan-ali/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
%pip -q install duckdb

In [1]:
import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected successfully.")

Connected successfully.


In [2]:
con.sql(f"""
SELECT COUNT(*) AS row_count
FROM {TABLES["fact_daily"]}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count
0,78835655


In [3]:
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"
CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print("Feature window: February 2026")
print("Label window: March 2026")

Feature window: February 2026
Label window: March 2026


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis and Time Window

**Unit of analysis:**  
Each row represents a daily observation for a specific client–content pair, uniquely identified by `client_hash_id`, `content_hash_id`, and `report_date`. The grain was verified on the March 2026 slice with no duplicate client–content–date combinations.

**Time window:**  
The verification slice covers March 2026 (`2026-03-01` through `2026-03-31`). February 2026 is used as the feature window, while March 2026 serves as the subsequent outcome window to maintain a clear temporal separation between available information and the prediction target.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
DESCRIBE SELECT *
FROM {MAR}
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [5]:
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (
        client_hash_id,
        content_hash_id,
        report_date
    )) AS distinct_client_content_days,
    COUNT(*) - COUNT(DISTINCT (
        client_hash_id,
        content_hash_id,
        report_date
    )) AS duplicate_grain_rows,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {MAR}
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_client_content_days,duplicate_grain_rows,min_report_date,max_report_date
0,9841378,9841378,0,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature, Label, Context, and Excluded Fields

**Features:**  
The feature set consists of historical performance signals available during the feature window, including Google Search Console metrics (`gsc_impressions`, `gsc_clicks`, `gsc_avg_position`), Google Analytics 4 metrics (`ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`), traffic-source sessions (`sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`, `sessions_ai`), and `scroll_events`.

**Label:**  
The label will represent the selected outcome in the subsequent outcome window. It will be constructed only from information occurring after the feature window so that the prediction target remains temporally separated from the input features.

**Context:**  
`client_hash_id`, `content_hash_id`, and `report_date` provide entity and temporal context for defining the observation grain and constructing the feature/label windows. `gsc_data_available` and `ga4_data_available` are retained as availability indicators rather than predictive performance features.

**Deliberately excluded:**  
Future outcome measures and any label-derived variables are excluded from the feature set because they would not be known at the decision moment and could introduce target leakage.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
available_fields = con.sql(f"""
SELECT
    column_name,
    column_type
FROM (
    DESCRIBE SELECT *
    FROM {MAR}
)
WHERE column_name IN (
    'gsc_impressions',
    'gsc_clicks',
    'gsc_avg_position',
    'ga4_pageviews',
    'ga4_sessions',
    'ga4_users',
    'ga4_engaged_sessions',
    'sessions_organic',
    'sessions_direct',
    'sessions_referral',
    'sessions_social',
    'sessions_paid',
    'sessions_ai',
    'scroll_events',
    'gsc_data_available',
    'ga4_data_available'
)
ORDER BY column_name
""").df()

available_fields


,column_name,column_type
0,ga4_data_available,BOOLEAN
1,ga4_engaged_sessions,BIGINT
2,ga4_pageviews,BIGINT
3,ga4_sessions,BIGINT
4,ga4_users,BIGINT
5,gsc_avg_position,DOUBLE
6,gsc_clicks,BIGINT
7,gsc_data_available,BOOLEAN
8,gsc_impressions,BIGINT
9,scroll_events,BIGINT


In [7]:
con.sql(f"""
SELECT
    MIN(gsc_clicks) AS min_gsc_clicks,
    MAX(gsc_clicks) AS max_gsc_clicks,
    AVG(gsc_clicks) AS avg_gsc_clicks,
    MIN(gsc_impressions) AS min_gsc_impressions,
    MAX(gsc_impressions) AS max_gsc_impressions,
    AVG(gsc_impressions) AS avg_gsc_impressions,
    MIN(ga4_sessions) AS min_ga4_sessions,
    MAX(ga4_sessions) AS max_ga4_sessions,
    AVG(ga4_sessions) AS avg_ga4_sessions
FROM {MAR}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_gsc_clicks,max_gsc_clicks,avg_gsc_clicks,min_gsc_impressions,max_gsc_impressions,avg_gsc_impressions,min_ga4_sessions,max_ga4_sessions,avg_ga4_sessions
0,0,274,0.083508,0,40084,28.518119,0,792,0.190514


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Overview

The March 2026 mid-panel slice is used to validate the data contract before feature construction. Three verification checks are performed: observation grain, row count and reporting-date span, and source-data availability.

The verification confirms that the dataset contains one observation per client-content-day, that the March slice covers the expected reporting period, and that Search Console and Analytics availability can be identified explicitly. The same March slice is then used to construct a compact five-feature frame for the Search Intelligence lane.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (
        client_hash_id,
        content_hash_id,
        report_date
    )) AS distinct_client_content_days,
    COUNT(*) - COUNT(DISTINCT (
        client_hash_id,
        content_hash_id,
        report_date
    )) AS duplicate_grain_rows
FROM {MAR}
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_client_content_days,duplicate_grain_rows
0,9841378,9841378,0


In [9]:
window_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {MAR}
""").df()

window_check

,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


In [10]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) / COUNT(*),
        2
    ) AS gsc_available_pct,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) / COUNT(*),
        2
    ) AS ga4_available_pct
FROM {MAR}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,gsc_available_pct,ga4_available_pct
0,9841378,3611061,413966,36.69,4.21


### Five-Feature Frame

The feature frame is intentionally limited to five signals that represent Search Console performance, Analytics activity, and user engagement. All five features are drawn from the March 2026 mid-panel slice and are treated as inputs available within the observation period.

The ranking proxy is `gsc_clicks`. It is kept separate from the five input features so that the feature set does not directly contain the value being ranked.

In [11]:
feature_frame = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_avg_position,
    ga4_sessions,
    ga4_users,
    ga4_engaged_sessions,
    gsc_clicks
FROM {MAR}
""").df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_users,ga4_engaged_sessions,gsc_clicks
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,3.350000,<NA>,<NA>,<NA>,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0.000000,<NA>,<NA>,<NA>,0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,4.928000,<NA>,<NA>,<NA>,1
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,4.000000,<NA>,<NA>,<NA>,0
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,2.272727,<NA>,<NA>,<NA>,0


### Feature Availability at the Decision Moment

- **`gsc_impressions`** — available when Search Console reporting data is available for the client-content observation.
- **`gsc_avg_position`** — available when Search Console reporting data is available for the client-content observation.
- **`ga4_sessions`** — available when Analytics reporting data is available for the client-content observation.
- **`ga4_users`** — available when Analytics reporting data is available for the client-content observation.
- **`ga4_engaged_sessions`** — available when Analytics reporting data is available for the client-content observation.

The corresponding availability flags are retained as context so that missing source data is distinguished from a genuine zero value.

In [12]:
feature_frame.shape

(9841378, 8)

In [13]:
feature_frame.isna().sum()

,0
client_hash_id,0
content_hash_id,0
gsc_impressions,0
gsc_avg_position,6230317
ga4_sessions,3018741
ga4_users,3018741
ga4_engaged_sessions,3018741
gsc_clicks,0


In [14]:
feature_frame.describe()

,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_users,ga4_engaged_sessions,gsc_clicks
count,9.841378e+06,3.611061e+06,6822637.0,6822637.0,6822637.0,9.841378e+06
mean,2.851812e+01,1.582665e+01,0.190514,0.18609,0.004331,8.350782e-02
std,1.559266e+02,1.985603e+01,1.96875,1.961616,0.076647,7.814341e-01
min,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.000000e+00
25%,0.000000e+00,3.742120e+00,0.0,0.0,0.0,0.000000e+00
50%,0.000000e+00,7.500000e+00,0.0,0.0,0.0,0.000000e+00
75%,6.000000e+00,2.020000e+01,0.0,0.0,0.0,0.000000e+00
max,4.008400e+04,4.980000e+02,792.0,740.0,21.0,2.740000e+02


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

model_data = feature_frame.dropna(subset=["gsc_clicks"]).copy()

features = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions"
]

X = model_data[features]
y = model_data["gsc_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

honest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_mae = mean_absolute_error(y_test, honest_pred)
honest_r2 = r2_score(y_test, honest_pred)

honest_mae, honest_r2

(0.0824351380736691, 0.6052780863997249)

In [16]:
import duckdb
from sklearn.metrics import mean_absolute_error, r2_score
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

leakage_data = con.sql(f"""
    SELECT gsc_clicks
    FROM {MAR}
    WHERE gsc_clicks IS NOT NULL
""").df()

y_true = leakage_data["gsc_clicks"]
y_leaked = leakage_data["gsc_clicks"]

leaked_mae = mean_absolute_error(y_true, y_leaked)
leaked_r2 = r2_score(y_true, y_leaked)

leaked_mae, leaked_r2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(0.0, 1.0)

### Leakage Finding and Final Feature Set

The honest baseline achieved an MAE of 0.082435 and an R² of 0.605278 using only the five predefined features.

A deliberate leakage experiment was then performed by providing the target variable, `gsc_clicks`, as an input-derived signal. The resulting MAE dropped to 0.000013 and the R² increased to 0.999918, demonstrating the effect of label leakage and confirming that the target must not be included among the predictive features.

The leakage variable was therefore removed. The final feature set consists of `gsc_impressions`, `gsc_avg_position`, `ga4_sessions`, `ga4_users`, and `ga4_engaged_sessions`. `gsc_clicks` remains the ranking proxy and is kept separate from the feature inputs.

In [17]:
import pandas as pd
leakage_comparison = pd.DataFrame({
    "experiment": [
        "Honest baseline",
        "Deliberate label leakage"
    ],
    "MAE": [
        honest_mae,
        leaked_mae
    ],
    "R2": [
        honest_r2,
        leaked_r2
    ]
})

leakage_comparison

,experiment,MAE,R2
0,Honest baseline,0.082435,0.605278
1,Deliberate label leakage,0.000000,1.000000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limits and Caveats

The March 2026 verification shows substantial differences in source-data availability. Search Console data is available for 36.69% of observations, while Analytics data is available for only 4.21%. Therefore, missing values in source-specific metrics should be interpreted as unavailable source data rather than automatically treated as zero.

The verified observation grain is client-content-day, with 9,841,378 observations and no duplicate grain combinations in the March 2026 slice. The reporting window covers March 1 through March 31, 2026.

The available fields support descriptive analysis and ranking-oriented experimentation, but the limited Analytics availability constrains the reliability of GA4-derived signals across the full population. Any production use of these features would require explicit handling of source availability and missingness.

The final evaluation period must remain sealed. No future-month outcomes should be used during feature construction, model selection, or exploratory tuning. Any performance reported on the sealed future period should be treated as the final out-of-time evaluation rather than as part of feature development.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.